<a href="https://colab.research.google.com/github/EnzoAA004/PFI_MVPTest_Enzo_AImodule/blob/enzo%2Fp10-8-clinical-expansion-preflight/75_P10_8_multiframe_hernia_review_no_retraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 75 — P10.8: revisión multiframe de hernia discal — sin reentrenamiento

Este notebook define y audita un **protocolo de revisión multiframe** para hernia discal.
El objetivo es documentar qué evidencia visual debería poder revisar un profesional
entre sagital/parasagital y axial, y verificar qué fuentes de datos están realmente
disponibles para sostener esa revisión.

**No crea un nuevo modelo de hernia.**

Reglas:

- no entrena;
- no carga ni deserializa `.pt`;
- no modifica checkpoints;
- no abre tests sellados;
- no crea ground truth clínico;
- no calcula probabilidades;
- no muestra barras de probabilidad;
- no fusiona pacientes de cohortes distintas;
- no habilita diagnóstico autónomo;
- la tarea `disc_herniation` continúa cerrada para reentrenamiento.


In [1]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Entorno no Colab")


Mounted at /content/drive


In [2]:
from __future__ import annotations

import hashlib
import json
import os
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd

ROOT = Path(
    os.getenv(
        "PFI_ROOT",
        "/content/drive/MyDrive/PFI_MVP",
    )
)
PREF = Path(
    os.getenv(
        "PFI_P10_8_PREFLIGHT_ROOT",
        str(
            ROOT
            / "results"
            / "P10_8_clinical_expansion_preflight"
        ),
    )
)
N73_ROOT = Path(
    os.getenv(
        "PFI_P10_8_NOTEBOOK73_ROOT",
        str(PREF / "viability_gate"),
    )
)
N74_ROOT = Path(
    os.getenv(
        "PFI_P10_8_NOTEBOOK74_ROOT",
        str(PREF / "geometry_measurement_protocol"),
    )
)
OUT = Path(
    os.getenv(
        "PFI_P10_8_NOTEBOOK75_ROOT",
        str(PREF / "multiframe_hernia_review"),
    )
)

MARKER_73 = N73_ROOT / "NOTEBOOK_73_COMPLETE.json"
CLOSED_73 = N73_ROOT / "closed_no_retraining_v1.csv"
MARKER_74 = N74_ROOT / "NOTEBOOK_74_COMPLETE.json"

for required in (MARKER_73, CLOSED_73, MARKER_74):
    if not required.is_file():
        raise FileNotFoundError(
            f"Falta entrada requerida: {required}"
        )

marker_73 = json.loads(
    MARKER_73.read_text(encoding="utf-8")
)
marker_74 = json.loads(
    MARKER_74.read_text(encoding="utf-8")
)
closed_73 = pd.read_csv(CLOSED_73)

if marker_73.get("status") != "NOTEBOOK_73_COMPLETE":
    raise RuntimeError("Notebook 73 no está cerrado.")

if marker_74.get("status") != "NOTEBOOK_74_COMPLETE":
    raise RuntimeError("Notebook 74 no está cerrado.")

if marker_74.get("schemaVersion") != (
    "pfi.p10-8.notebook-74-complete.v1"
):
    raise RuntimeError(
        "Notebook 74 no tiene el schemaVersion esperado."
    )

for label, marker in [
    ("Notebook 73", marker_73),
    ("Notebook 74", marker_74),
]:
    if marker.get("trainingExecuted") is not False:
        raise RuntimeError(
            f"{label} no declara trainingExecuted=false"
        )
    if marker.get("weightsDeserialized") is not False:
        raise RuntimeError(
            f"{label} no declara weightsDeserialized=false"
        )
    if marker.get("trainingAuthorized") is not False:
        raise RuntimeError(
            f"{label} no mantiene trainingAuthorized=false"
        )

hernia_rows = closed_73[
    closed_73["findingType"].astype(str)
    == "disc_herniation"
].copy()

if len(hernia_rows) != 1:
    raise RuntimeError(
        "Se esperaba exactamente una fila cerrada "
        "para disc_herniation en Notebook 73."
    )

hernia_row = hernia_rows.iloc[0]

if not str(
    hernia_row["gateDecision"]
).startswith("CLOSED_NO_RETRAINING"):
    raise RuntimeError(
        "disc_herniation no está protegida contra "
        "reentrenamiento."
    )

if str(
    hernia_row["targetNextStep"]
) != "NOTEBOOK_75_REVIEW_ONLY_NO_RETRAINING":
    raise RuntimeError(
        "Notebook 73 no derivó hernia al flujo de "
        "revisión multiframe esperado."
    )

print("Notebooks 73 y 74 verificados.")
print("disc_herniation: cerrada para reentrenamiento.")
print("Salida Notebook 75:", OUT)


Notebooks 73 y 74 verificados.
disc_herniation: cerrada para reentrenamiento.
Salida Notebook 75: /content/drive/MyDrive/PFI_MVP/results/P10_8_clinical_expansion_preflight/multiframe_hernia_review


In [3]:
def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(
        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False,
            sort_keys=True,
        )
        + "\n",
        encoding="utf-8",
    )
    os.replace(temp, path)

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)
    return digest.hexdigest()

def sha_text(value: str) -> str:
    return hashlib.sha256(
        str(value).encode("utf-8")
    ).hexdigest()

input_hashes = {
    "NOTEBOOK_73_COMPLETE.json":
        sha256_file(MARKER_73),
    "closed_no_retraining_v1.csv":
        sha256_file(CLOSED_73),
    "NOTEBOOK_74_COMPLETE.json":
        sha256_file(MARKER_74),
}

display(
    pd.DataFrame(
        [
            {
                "inputName": key,
                "sha256": value,
            }
            for key, value in input_hashes.items()
        ]
    )
)


,inputName,sha256
0,NOTEBOOK_73_COMPLETE.json,f87f266eef495d0e1387820cdb0997a48eb64b33303a15...
1,closed_no_retraining_v1.csv,409a9a88ad87a127dc45836660071328f95533b4c1c322...
2,NOTEBOOK_74_COMPLETE.json,b017e19444c2cd05b1771f7d5a97aac9592c27053fee09...


## Contrato clínico-operativo de revisión multiframe

La revisión de una sospecha de hernia no se reduce a un único recorte ni a una única
vista. Para P10.8 se documenta el siguiente flujo de apoyo:

1. imagen original primero;
2. selección/revisión del nivel por un profesional;
3. revisión sagital central y parasagital cuando esté disponible;
4. revisión de los cortes sagitales adyacentes que el profesional considere relevantes;
5. revisión axial correspondiente al nivel;
6. revisión de los cortes axiales adyacentes relevantes;
7. comparación visual entre planos;
8. overlays solo bajo demanda;
9. registro separado de la revisión profesional.

El protocolo **no fija una cantidad universal de cortes**: la suficiencia del conjunto
multiframe queda sujeta a cobertura anatómica y revisión profesional.


In [4]:
review_requirements = pd.DataFrame(
    [
        {
            "ruleId": "H01",
            "stage": "presentation",
            "requirement":
                "Present original image before overlays.",
            "blocking": True,
        },
        {
            "ruleId": "H02",
            "stage": "level_selection",
            "requirement":
                "Disc level must be selected or reviewed "
                "by a professional.",
            "blocking": True,
        },
        {
            "ruleId": "H03",
            "stage": "sagittal_review",
            "requirement":
                "Review central sagittal and available "
                "parasagittal frames relevant to the level.",
            "blocking": True,
        },
        {
            "ruleId": "H04",
            "stage": "sagittal_review",
            "requirement":
                "Adjacent sagittal frames judged relevant "
                "must remain available.",
            "blocking": True,
        },
        {
            "ruleId": "H05",
            "stage": "axial_review",
            "requirement":
                "Review axial frame corresponding to the "
                "disc level.",
            "blocking": True,
        },
        {
            "ruleId": "H06",
            "stage": "axial_review",
            "requirement":
                "Adjacent axial frames judged relevant "
                "must remain available.",
            "blocking": True,
        },
        {
            "ruleId": "H07",
            "stage": "cross_plane_review",
            "requirement":
                "Sagittal/axial association must be "
                "professionally reviewed until automatic "
                "cross-reference is validated.",
            "blocking": True,
        },
        {
            "ruleId": "H08",
            "stage": "overlay",
            "requirement":
                "Overlays are optional and shown on demand "
                "after the original image.",
            "blocking": False,
        },
        {
            "ruleId": "H09",
            "stage": "result",
            "requirement":
                "No probability bar or severity category "
                "is produced by this review-only protocol.",
            "blocking": True,
        },
        {
            "ruleId": "H10",
            "stage": "audit",
            "requirement":
                "Professional review decision is stored "
                "separately from immutable model outputs.",
            "blocking": True,
        },
    ]
)

display(review_requirements)


,ruleId,stage,requirement,blocking
0,H01,presentation,Present original image before overlays.,True
1,H02,level_selection,Disc level must be selected or reviewed by a p...,True
2,H03,sagittal_review,Review central sagittal and available parasagi...,True
3,H04,sagittal_review,Adjacent sagittal frames judged relevant must ...,True
4,H05,axial_review,Review axial frame corresponding to the disc l...,True
5,H06,axial_review,Adjacent axial frames judged relevant must rem...,True
6,H07,cross_plane_review,Sagittal/axial association must be professiona...,True
7,H08,overlay,Overlays are optional and shown on demand afte...,False
8,H09,result,No probability bar or severity category is pro...,True
9,H10,audit,Professional review decision is stored separat...,True


In [5]:
frame_roles = pd.DataFrame(
    [
        {
            "plane": "sagittal",
            "frameRole": "central",
            "requiredByProtocol": True,
            "automaticSelectionValidated": False,
        },
        {
            "plane": "sagittal",
            "frameRole": "parasagittal_left_or_right",
            "requiredByProtocol":
                "when_anatomically_relevant_and_available",
            "automaticSelectionValidated": False,
        },
        {
            "plane": "sagittal",
            "frameRole": "adjacent_relevant",
            "requiredByProtocol":
                "professional_judgement",
            "automaticSelectionValidated": False,
        },
        {
            "plane": "axial",
            "frameRole": "level_corresponding",
            "requiredByProtocol": True,
            "automaticSelectionValidated": False,
        },
        {
            "plane": "axial",
            "frameRole": "adjacent_relevant",
            "requiredByProtocol":
                "professional_judgement",
            "automaticSelectionValidated": False,
        },
    ]
)

display(frame_roles)


,plane,frameRole,requiredByProtocol,automaticSelectionValidated
0,sagittal,central,True,False
1,sagittal,parasagittal_left_or_right,when_anatomically_relevant_and_available,False
2,sagittal,adjacent_relevant,professional_judgement,False
3,axial,level_corresponding,True,False
4,axial,adjacent_relevant,professional_judgement,False


## Auditoría de disponibilidad de fuentes

El notebook no intenta emparejar cohortes diferentes. Solamente registra si existen
fuentes sagitales y axiales conocidas, si su esquema permite identificar casos/series/
cortes y si hay evidencia de un **mismo paciente y nivel en ambos planos**.

La coincidencia de números o nombres entre datasets nunca se considera prueba de que
sea el mismo paciente.


In [6]:
# Fuente axial conocida del trabajo previo E7/Al-Kafri.
E7_ROOT = (
    ROOT
    / "results"
    / "E7_alkafri_axial_curated_subset"
)
AXIAL_INDEX = (
    E7_ROOT
    / "E7_alkafri_axial_image_case_index.csv"
)

# Búsqueda limitada a carpetas P10.7 esperables.
# No se recorre todo Google Drive.
p10_7_roots = []

explicit_p10_7 = os.getenv(
    "PFI_P10_7_RESULTS_ROOT",
    "",
).strip()

if explicit_p10_7:
    candidate = Path(explicit_p10_7)
    if candidate.is_dir():
        p10_7_roots.append(candidate)

results_root = ROOT / "results"
if results_root.is_dir():
    for candidate in results_root.glob("*P10_7*"):
        if candidate.is_dir():
            p10_7_roots.append(candidate)
    for candidate in results_root.glob("*p10_7*"):
        if candidate.is_dir():
            p10_7_roots.append(candidate)
    for candidate in results_root.glob("*spider*"):
        if candidate.is_dir():
            p10_7_roots.append(candidate)

# Deduplicación por ruta resuelta.
deduped = []
seen = set()

for path in p10_7_roots:
    resolved = str(path.resolve())
    if resolved not in seen:
        seen.add(resolved)
        deduped.append(path)

p10_7_roots = deduped

print("Raíces P10.7 candidatas:", len(p10_7_roots))
for path in p10_7_roots:
    print("-", path)

print("Índice axial E7 existe:", AXIAL_INDEX.is_file())


Raíces P10.7 candidatas: 3
- /content/drive/MyDrive/PFI_MVP/results/P10_7_spider_degenerative
- /content/drive/MyDrive/PFI_MVP/results/GCS_spider_smoke_test
- /content/drive/MyDrive/PFI_MVP/results/GCS_spider_final_training_v1
Índice axial E7 existe: True


In [7]:
# Auditoría de esquemas tabulares sin exportar identificadores.
interesting = re.compile(
    r"(patient|case|study|series|level|disc|"
    r"plane|view|modality|slice|frame|path|image)",
    re.IGNORECASE,
)

schema_rows = []
p10_7_csv_candidates = []

for root in p10_7_roots:
    for path in root.rglob("*.csv"):
        if path.stat().st_size > 100 * 1024 * 1024:
            continue
        p10_7_csv_candidates.append(path)

# Cap conservador.
p10_7_csv_candidates = sorted(
    set(p10_7_csv_candidates),
    key=lambda p: sha_text(str(p)),
)[:100]

for path in p10_7_csv_candidates:
    try:
        frame = pd.read_csv(
            path,
            nrows=100,
            dtype=str,
            low_memory=False,
        )
        columns = [str(c) for c in frame.columns]
        matching = [
            c for c in columns if interesting.search(c)
        ]

        schema_rows.append(
            {
                "sourceFamily": "P10_7_SPIDER",
                "fileName": path.name,
                "filePathHash": sha_text(str(path)),
                "columnCount": len(columns),
                "matchingFieldCount": len(matching),
                "matchingFields":
                    "|".join(sorted(matching))[:4000],
                "readSucceeded": True,
                "readErrorType": None,
            }
        )
    except Exception as exc:
        schema_rows.append(
            {
                "sourceFamily": "P10_7_SPIDER",
                "fileName": path.name,
                "filePathHash": sha_text(str(path)),
                "columnCount": None,
                "matchingFieldCount": None,
                "matchingFields": "",
                "readSucceeded": False,
                "readErrorType": type(exc).__name__,
            }
        )

if AXIAL_INDEX.is_file():
    try:
        axial_preview = pd.read_csv(
            AXIAL_INDEX,
            nrows=100,
            dtype=str,
            low_memory=False,
        )
        columns = [
            str(c) for c in axial_preview.columns
        ]
        matching = [
            c for c in columns if interesting.search(c)
        ]

        schema_rows.append(
            {
                "sourceFamily": "E7_ALKAFRI_AXIAL",
                "fileName": AXIAL_INDEX.name,
                "filePathHash":
                    sha_text(str(AXIAL_INDEX)),
                "columnCount": len(columns),
                "matchingFieldCount": len(matching),
                "matchingFields":
                    "|".join(sorted(matching))[:4000],
                "readSucceeded": True,
                "readErrorType": None,
            }
        )
    except Exception as exc:
        schema_rows.append(
            {
                "sourceFamily": "E7_ALKAFRI_AXIAL",
                "fileName": AXIAL_INDEX.name,
                "filePathHash":
                    sha_text(str(AXIAL_INDEX)),
                "columnCount": None,
                "matchingFieldCount": None,
                "matchingFields": "",
                "readSucceeded": False,
                "readErrorType": type(exc).__name__,
            }
        )

schema_audit = pd.DataFrame(
    schema_rows,
    columns=[
        "sourceFamily",
        "fileName",
        "filePathHash",
        "columnCount",
        "matchingFieldCount",
        "matchingFields",
        "readSucceeded",
        "readErrorType",
    ],
)

display(schema_audit)


,sourceFamily,fileName,filePathHash,columnCount,matchingFieldCount,matchingFields,readSucceeded,readErrorType
0,P10_7_SPIDER,spider_pair_split.csv,0339ecc32a17f7df96eeefaaebba87e0ac56e89eedb9cb...,6,3,image_local_path|mask_local_path|patient_id,True,None
1,P10_7_SPIDER,spider_patient_split.csv,03bba4a0a2fbac90bf2a8c24d1191f72baccf985db467f...,2,1,patient_id,True,None
2,P10_7_SPIDER,spider_cache_receipt.csv,057eb45dc4dc55fca0a53e75165466c9135eb73ab097ef...,9,5,image_cache_status|image_crc32c|image_local_pa...,True,None
3,P10_7_SPIDER,patient_split_v1.csv,105e16158e895fc8a4702053c9460fe47e05051e7439d9...,3,1,Patient,True,None
4,P10_7_SPIDER,disc_level_crop_failures_v1.csv,207a91c7aec3aac298ffdda0466c9609ba48d19262b44b...,4,2,Patient|modality_failures_json,True,None
5,P10_7_SPIDER,smoke_batch_history.csv,2506fb6c50446cb341bdc19f28241cbd9a20936494d498...,4,0,,True,None
6,P10_7_SPIDER,training_history.csv,2618c773d63ae379a7b8179f77695d9f1ec92be86b5c40...,11,0,,True,None
7,P10_7_SPIDER,selected_smoke_pairs.csv,49870c9279ba7fbf55874bc1e6b7d75f6980ef88d79bdf...,10,4,image_crc32c|image_gcs_uri|image_size_bytes|pa...,True,None
8,P10_7_SPIDER,disc_level_manifest_v1.csv,546428a954543a41c385ae3da54c5a87d096998965f707...,17,6,Patient|crop_path|disc_bulging|disc_herniation...,True,None
9,P10_7_SPIDER,spider_slice_records.csv,74e6d4861573c61e37763b6e8b1a84a3f77d1f84f17c77...,8,5,image_local_path|mask_local_path|patient_id|sl...,True,None


In [8]:
# Conteos agregados de disponibilidad.
axial_rows = None
axial_cases = None

if AXIAL_INDEX.is_file():
    try:
        axial_full = pd.read_csv(
            AXIAL_INDEX,
            dtype=str,
            low_memory=False,
        )
        axial_rows = int(len(axial_full))

        candidate_case_columns = [
            c
            for c in axial_full.columns
            if re.search(
                r"(case|patient)",
                str(c),
                re.IGNORECASE,
            )
        ]

        if candidate_case_columns:
            case_col = candidate_case_columns[0]
            axial_cases = int(
                axial_full[case_col]
                .dropna()
                .astype(str)
                .nunique()
            )
    except Exception:
        pass

p10_7_tabular_source_found = bool(
    len(p10_7_csv_candidates) > 0
)
axial_source_found = AXIAL_INDEX.is_file()

source_readiness = pd.DataFrame(
    [
        {
            "sourceFamily": "P10_7_SPIDER",
            "role": "existing_hernia_model_context",
            "tabularSourceFound":
                p10_7_tabular_source_found,
            "rowCount": None,
            "uniqueCaseCount": None,
            "samePatientCrossPlanePairingValidated": False,
            "crossCohortPairingAllowed": False,
        },
        {
            "sourceFamily": "E7_ALKAFRI_AXIAL",
            "role": "axial_source_context",
            "tabularSourceFound":
                axial_source_found,
            "rowCount": axial_rows,
            "uniqueCaseCount": axial_cases,
            "samePatientCrossPlanePairingValidated": False,
            "crossCohortPairingAllowed": False,
        },
    ]
)

display(source_readiness)


,sourceFamily,role,tabularSourceFound,rowCount,uniqueCaseCount,samePatientCrossPlanePairingValidated,crossCohortPairingAllowed
0,P10_7_SPIDER,existing_hernia_model_context,True,NaN,NaN,False,False
1,E7_ALKAFRI_AXIAL,axial_source_context,True,8150.0,185.0,False,False


In [9]:
pairing_assessment = pd.DataFrame(
    [
        {
            "assessmentId":
                "SPIDER_SAGITTAL_TO_ALKAFRI_AXIAL",
            "sagittalSourceFamily":
                "P10_7_SPIDER",
            "axialSourceFamily":
                "E7_ALKAFRI_AXIAL",
            "sameCohortValidated": False,
            "samePatientIdentityValidated": False,
            "sameDiscLevelValidated": False,
            "sameStudyValidated": False,
            "sameSeriesGeometryValidated": False,
            "pairingAllowed": False,
            "reason":
                "Different source families must not be "
                "paired by coincident names or numbers.",
        }
    ]
)

full_multiframe_same_patient_validation = bool(
    pairing_assessment[
        "pairingAllowed"
    ].all()
)

display(pairing_assessment)

print(
    "Full same-patient sagittal+axial validation:",
    full_multiframe_same_patient_validation,
)


,assessmentId,sagittalSourceFamily,axialSourceFamily,sameCohortValidated,samePatientIdentityValidated,sameDiscLevelValidated,sameStudyValidated,sameSeriesGeometryValidated,pairingAllowed,reason
0,SPIDER_SAGITTAL_TO_ALKAFRI_AXIAL,P10_7_SPIDER,E7_ALKAFRI_AXIAL,False,False,False,False,False,False,Different source families must not be paired b...


Full same-patient sagittal+axial validation: False


In [10]:
review_contract = {
    "schemaVersion":
        "pfi.p10-8.multiframe-hernia-review.v1",
    "findingType": "disc_herniation",
    "mode": "review_only_no_retraining",
    "retrainingAuthorized": False,
    "trainingAuthorized": False,
    "weightsDeserialized": False,
    "probabilityBarsAllowed": False,
    "severityClassificationAllowed": False,
    "notClinicalDiagnosis": True,
    "professionalReviewRequired": True,
    "originalImageFirst": True,
    "overlayOnDemand": True,
    "requiredReview": {
        "sagittal": {
            "central": True,
            "parasagittal":
                "when_relevant_and_available",
            "adjacentFrames":
                "professional_judgement",
        },
        "axial": {
            "levelCorresponding": True,
            "adjacentFrames":
                "professional_judgement",
        },
        "crossPlaneAssociation": {
            "automaticValidated": False,
            "professionalReviewRequired": True,
        },
    },
    "samePatientCrossPlanePairingValidated":
        full_multiframe_same_patient_validation,
    "crossCohortPairingAllowed": False,
    "clinicalInterpretation":
        "professional_review_only",
}

print(
    json.dumps(
        review_contract,
        indent=2,
        ensure_ascii=False,
    )
)


{
  "schemaVersion": "pfi.p10-8.multiframe-hernia-review.v1",
  "findingType": "disc_herniation",
  "mode": "review_only_no_retraining",
  "retrainingAuthorized": false,
  "trainingAuthorized": false,
  "weightsDeserialized": false,
  "probabilityBarsAllowed": false,
  "severityClassificationAllowed": false,
  "notClinicalDiagnosis": true,
  "professionalReviewRequired": true,
  "originalImageFirst": true,
  "overlayOnDemand": true,
  "requiredReview": {
    "sagittal": {
      "central": true,
      "parasagittal": "when_relevant_and_available",
      "adjacentFrames": "professional_judgement"
    },
    "axial": {
      "levelCorresponding": true,
      "adjacentFrames": "professional_judgement"
    },
    "crossPlaneAssociation": {
      "automaticValidated": false,
      "professionalReviewRequired": true
    }
  },
  "samePatientCrossPlanePairingValidated": false,
  "crossCohortPairingAllowed": false,
  "clinicalInterpretation": "professional_review_only"
}


In [11]:
OUT.mkdir(parents=True, exist_ok=True)

output_paths = {
    "reviewRequirements":
        OUT / "multiframe_review_requirements_v1.csv",
    "frameRoles":
        OUT / "multiframe_frame_roles_v1.csv",
    "schemaAudit":
        OUT / "multiframe_source_schema_audit_v1.csv",
    "sourceReadiness":
        OUT / "multiframe_source_readiness_v1.csv",
    "pairingAssessment":
        OUT / "cross_plane_pairing_assessment_v1.csv",
    "reviewContract":
        OUT / "multiframe_hernia_review_contract_v1.json",
    "inputHashes":
        OUT / "notebook73_74_input_hashes_v1.csv",
    "summary":
        OUT / "NOTEBOOK_75_SUMMARY.json",
}

review_requirements.to_csv(
    output_paths["reviewRequirements"],
    index=False,
)
frame_roles.to_csv(
    output_paths["frameRoles"],
    index=False,
)
schema_audit.to_csv(
    output_paths["schemaAudit"],
    index=False,
)
source_readiness.to_csv(
    output_paths["sourceReadiness"],
    index=False,
)
pairing_assessment.to_csv(
    output_paths["pairingAssessment"],
    index=False,
)
write_json(
    output_paths["reviewContract"],
    review_contract,
)
pd.DataFrame(
    [
        {
            "inputName": key,
            "sha256": value,
        }
        for key, value in input_hashes.items()
    ]
).to_csv(
    output_paths["inputHashes"],
    index=False,
)

pt_files = list(OUT.rglob("*.pt"))

if pt_files:
    raise RuntimeError(
        "La salida del Notebook 75 contiene .pt "
        "inesperados."
    )

summary = {
    "schemaVersion":
        "pfi.p10-8.notebook-75-summary.v1",
    "generatedAtUtc":
        datetime.now(timezone.utc).isoformat(),
    "findingType": "disc_herniation",
    "reviewMode":
        "multiframe_review_only_no_retraining",
    "trainingExecuted": False,
    "retrainingAuthorized": False,
    "trainingAuthorized": False,
    "weightsDeserialized": False,
    "internalTestAccessed": False,
    "officialHiddenTestAccessed": False,
    "patientIdentifiersExported": False,
    "clinicalGroundTruthCreated": False,
    "probabilityBarsAllowed": False,
    "severityClassificationAllowed": False,
    "professionalReviewRequired": True,
    "notClinicalDiagnosis": True,
    "reviewRequirementCount":
        int(len(review_requirements)),
    "frameRoleCount":
        int(len(frame_roles)),
    "p10_7TabularSourceFound":
        p10_7_tabular_source_found,
    "axialSourceFound":
        axial_source_found,
    "samePatientCrossPlanePairingValidated":
        full_multiframe_same_patient_validation,
    "crossCohortPairingAllowed": False,
    "fullProductMultiframeValidation": False,
    "outputPtFileCount": len(pt_files),
    "nextRequiredGate":
        "NOTEBOOK_76_T1_T2_AND_SAGITTAL_AXIAL_ALIGNMENT",
}

write_json(
    output_paths["summary"],
    summary,
)

marker = {
    **summary,
    "schemaVersion":
        "pfi.p10-8.notebook-75-complete.v1",
    "summarySchemaVersion":
        summary["schemaVersion"],
    "status": "NOTEBOOK_75_COMPLETE",
    "outputs": {
        key: str(value)
        for key, value in output_paths.items()
    },
}

write_json(
    OUT / "NOTEBOOK_75_COMPLETE.json",
    marker,
)

print(
    json.dumps(
        marker,
        indent=2,
        ensure_ascii=False,
    )
)
print("NOTEBOOK_75_COMPLETE")


{
  "schemaVersion": "pfi.p10-8.notebook-75-complete.v1",
  "generatedAtUtc": "2026-08-07T02:51:15.919316+00:00",
  "findingType": "disc_herniation",
  "reviewMode": "multiframe_review_only_no_retraining",
  "trainingExecuted": false,
  "retrainingAuthorized": false,
  "trainingAuthorized": false,
  "weightsDeserialized": false,
  "internalTestAccessed": false,
  "officialHiddenTestAccessed": false,
  "patientIdentifiersExported": false,
  "clinicalGroundTruthCreated": false,
  "probabilityBarsAllowed": false,
  "severityClassificationAllowed": false,
  "professionalReviewRequired": true,
  "notClinicalDiagnosis": true,
  "reviewRequirementCount": 10,
  "frameRoleCount": 5,
  "p10_7TabularSourceFound": true,
  "axialSourceFound": true,
  "samePatientCrossPlanePairingValidated": false,
  "crossCohortPairingAllowed": false,
  "fullProductMultiframeValidation": false,
  "outputPtFileCount": 0,
  "nextRequiredGate": "NOTEBOOK_76_T1_T2_AND_SAGITTAL_AXIAL_ALIGNMENT",
  "summarySchemaVersion"

## Interpretación obligatoria

`NOTEBOOK_75_COMPLETE` significa que el **protocolo de revisión** quedó documentado y
que las fuentes disponibles fueron auditadas. No significa que exista una validación
producto de hernia multiframe.

Mientras `samePatientCrossPlanePairingValidated=false`:

- no se debe afirmar que sagital y axial fueron fusionados para el mismo paciente;
- no se deben emparejar cohortes distintas por coincidencia de identificadores;
- no se debe reentrenar `disc_herniation`;
- la revisión multiframe permanece como flujo profesional de investigación;
- el siguiente paso es auditar T1/T2 y la alineación sagital–axial en Notebook 76.
